Creating Dataset

In [82]:
import pandas as pd
import random

# =========================================================
# 1. EXTENDED GROCERY PRODUCT LIST (100+ PRODUCTS)
# =========================================================
grocery_products = [
    # Produce
    "Bananas", "Apples", "Spinach", "Lettuce", "Avocado", "Onions", "Potatoes",
    "Sweet Potatoes", "Carrots", "Strawberries", "Blueberries", "Raspberries",
    "Garlic", "Tomatoes", "Broccoli", "Cauliflower", "Cucumber", "Zucchini",
    "Peppers", "Lemons", "Limes",
    # Dairy & Eggs
    "Milk", "Almond Milk", "Oat Milk", "Soy Milk", "Eggs", "Butter", "Yogurt",
    "Cheese", "Cream Cheese", "Sour Cream", "Cottage Cheese",
    # Meat & Seafood
    "Chicken Breast", "Chicken Thighs", "Ground Beef", "Bacon", "Turkey Bacon",
    "Salmon", "Tilapia", "Shrimp", "Pork Chops", "Deli Meat", "Whole Chicken",
    # Bakery
    "Bread", "Bagels", "Tortillas", "English Muffins", "Croissants", "Buns", "Sourdough",
    # Pantry & Grains
    "Olive Oil", "Canola Oil", "Vegetable Oil", "Flour", "Sugar", "Rice", "Pasta",
    "Peanut Butter", "Almond Butter", "Jam", "Honey", "Maple Syrup", "Beans",
    "Chickpeas", "Tomato Sauce",
    # Snacks
    "Chips", "Pretzels", "Almonds", "Cashews", "Popcorn", "Granola Bars",
    "Protein Bars", "Chocolate", "Fruit Snacks", "Jerky",
    # Beverages
    "Coffee", "Espresso", "Tea", "Orange Juice", "Apple Juice",
    "Sparkling Water", "Club Soda", "Energy Drink", "Soda",
    # Frozen
    "Frozen Pizza", "Frozen Vegetables", "Ice Cream", "Frozen Waffles",
    "Chicken Nuggets", "Frozen Berries", "Frozen Burritos",
    # Household
    "Paper Towels", "Toilet Paper", "Dish Soap", "Laundry Detergent",
    "Fabric Softener", "Trash Bags", "Aluminum Foil"
]

# =========================================================
# 2. REVIEW TEMPLATES — EXACTLY 10 PER RATING
# =========================================================
review_templates = {
    5: [
        "Absolutely fresh and high quality!",
        "Best value for money.",
        "Tastes amazing and very fresh.",
        "Exceeded my expectations.",
        "Will definitely buy again.",
        "Perfect condition upon delivery.",
        "Top-notch quality, highly recommended.",
        "Consistently great product.",
        "My family loved this product.",
        "Worth every penny."
    ],
    4: [
        "Very good product, would recommend.",
        "Fresh and tasty overall.",
        "Good quality for the price.",
        "Arrived on time and well packed.",
        "A reliable grocery item.",
        "No complaints, good purchase.",
        "Better than expected.",
        "Decent quality and freshness.",
        "Satisfied with this product.",
        "Would buy again occasionally."
    ],
    3: [
        "It was okay, nothing special.",
        "Average quality overall.",
        "Decent but not outstanding.",
        "The taste was just okay.",
        "Acceptable for the price.",
        "Not bad, not great either.",
        "Mediocre quality.",
        "Could be better.",
        "Does the job but nothing more.",
        "Fairly average experience."
    ],
    2: [
        "Not very impressed with the quality.",
        "Wasn't as fresh as expected.",
        "Packaging could be improved.",
        "Taste was slightly disappointing.",
        "Below average quality.",
        "Probably won't buy again.",
        "Not worth the price.",
        "Quality was inconsistent.",
        "Expected better freshness.",
        "Somewhat disappointed."
    ],
    1: [
        "Horrible quality, do not buy.",
        "Item was expired or near expiration.",
        "Tasted really bad.",
        "Completely unusable.",
        "Waste of money.",
        "Received damaged product.",
        "Very poor quality.",
        "Extremely disappointed.",
        "Never buying this again.",
        "Worst grocery purchase."
    ]
}

# =========================================================
# 3. GENERATE DATASET
# =========================================================
num_users = 500
num_rows = 10000
data = []

for _ in range(num_rows):
    user_id = f"U{random.randint(1, num_users)}"
    product = random.choice(grocery_products)
    rating = random.choices([1,2,3,4,5], weights=[5,10,20,30,35])[0]
    review = random.choice(review_templates[rating])

    data.append({
        "user_id": user_id,
        "Product_Name": product,
        "Review": review,
        "Rating": rating
    })

df = pd.DataFrame(data)
df.to_csv("grocery_products_dataset.csv", index=False)

print("Dataset created with users")
print(df.head())

Dataset created with users
  user_id Product_Name                            Review  Rating
0     U39         Soda     No complaints, good purchase.       4
1     U12    Sourdough  Perfect condition upon delivery.       5
2    U459  Ground Beef  Arrived on time and well packed.       4
3    U374       Apples                  Could be better.       3
4    U149  Raspberries     Decent quality and freshness.       4


Item Based Recommendation

Import Libraries and Load Dataset

In [83]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
df = pd.read_csv("grocery_products_dataset.csv")

Sentiment Strength

In [84]:
def sentiment_score(r):
    if r >= 4:
        return 1
    elif r == 3:
        return 0
    else:
        return -1

df['sentiment'] = df['Rating'].apply(sentiment_score)
df['weighted_rating'] = 0.7 * df['Rating'] + 0.3 * df['sentiment']

Filter Low-Frequency Items

In [85]:
item_counts = df['Product_Name'].value_counts()
popular_items = item_counts[item_counts >= 20].index
df = df[df['Product_Name'].isin(popular_items)]

Train–Test Split

In [86]:
train, test = train_test_split(df, test_size=0.2, random_state=42)

Item–User Matrix

In [87]:
item_user = train.pivot_table(
    index='Product_Name',
    columns='user_id',
    values='weighted_rating',
    aggfunc='mean'
)

Item Similarity (Cosine)

In [88]:
item_similarity = pd.DataFrame(
    cosine_similarity(item_user.fillna(0)),
    index=item_user.index,
    columns=item_user.index
)

Item Recommendation Function

In [89]:
def item_based_recommend(user_id, top_n=5, top_k_items=5):
    user_items = train[train.user_id == user_id]['Product_Name'].unique()
    scores = {}

    for item in user_items:
        if item not in item_similarity.index:
            continue

        similar_items = item_similarity[item].sort_values(ascending=False)[1:top_k_items+1]

        for sim_item, sim_score in similar_items.items():
            if sim_item in user_items:
                continue
            scores[sim_item] = scores.get(sim_item, 0) + sim_score

    return sorted(scores, key=scores.get, reverse=True)[:top_n]

Evaluation

In [90]:
def evaluate_item_based(top_n=5):
    accs, precs, recs = [], [], []

    for user in test['user_id'].unique()[:100]:
        actual = set(test[test.user_id == user]['Product_Name'])
        if user not in train['user_id'].values:
            continue

        recommended = set(item_based_recommend(user, top_n))
        if not recommended:
            continue

        tp = len(recommended & actual)

        precision = tp / top_n
        recall = tp / len(actual)
        accuracy = tp / len(recommended | actual)

        accs.append(accuracy)
        precs.append(precision)
        recs.append(recall)

    return np.mean(accs), np.mean(precs), np.mean(recs)

In [91]:
accuracy, precision, recall = evaluate_item_based()

print("IMPROVED ITEM-BASED COLLABORATIVE FILTERING")
print("Accuracy :", round(accuracy, 3))
print("Precision:", round(precision, 3))
print("Recall   :", round(recall, 3))

IMPROVED ITEM-BASED COLLABORATIVE FILTERING
Accuracy : 0.027
Precision: 0.05
Recall   : 0.05


Save the Model

In [92]:
import joblib

# Save model components
joblib.dump(item_similarity, "item_similarity.pkl")
joblib.dump(item_user.index.tolist(), "item_list.pkl")
joblib.dump(train['user_id'].unique().tolist(), "train_users.pkl")
joblib.dump(popular_items.tolist(), "popular_items.pkl")

# Save configuration
config = {
    "top_n": 5,
    "top_k_items": 5,
    "sentiment_weight": 0.3,
    "rating_weight": 0.7
}
joblib.dump(config, "item_cf_config.pkl")

print("Item-based recommendation model saved successfully!")

Item-based recommendation model saved successfully!


Model Prediction

In [124]:
import pandas as pd
import numpy as np
import joblib

# -----------------------------
# Load trained item-based model
# -----------------------------
item_similarity = joblib.load("item_similarity.pkl")   # DataFrame
item_list = joblib.load("item_list.pkl")               # list of items
config = joblib.load("item_cf_config.pkl")             # config dict

print("Item-based model loaded")

# -----------------------------
# NEW USER DATA (external input)
# -----------------------------
new_user_data = pd.DataFrame([
    {"Product_Name": "Butter", "Rating": 5}
])

# -----------------------------
# Sentiment function
# -----------------------------
def sentiment_score(rating):
    if rating >= 4:
        return 1
    elif rating == 3:
        return 0
    else:
        return -1

# -----------------------------
# Build item preference scores
# -----------------------------
scores = {}

for _, row in new_user_data.iterrows():
    item = row["Product_Name"]
    if item not in item_similarity.index:
        continue

    # sentiment-weighted interaction
    weight = (
        config["rating_weight"] * row["Rating"] +
        config["sentiment_weight"] * sentiment_score(row["Rating"])
    )

    # get top-K similar items
    similar_items = (
        item_similarity[item]
        .sort_values(ascending=False)
        .iloc[1:config["top_k_items"]+1]
    )

    for sim_item, sim_score in similar_items.items():
        if sim_item in new_user_data["Product_Name"].values:
            continue
        scores[sim_item] = scores.get(sim_item, 0) + sim_score * weight

# -----------------------------
# Final recommendations
# -----------------------------
recommendations = sorted(
    scores, key=scores.get, reverse=True
)[:config["top_n"]]

# -----------------------------
# OUTPUT
# -----------------------------
print("\n NEW USER INPUT:")
print(new_user_data)

print("\n ITEM-BASED RECOMMENDATIONS:")
for i, item in enumerate(recommendations, 1):
    print(f"{i}. {item}")

Item-based model loaded

 NEW USER INPUT:
  Product_Name  Rating
0       Butter       5

 ITEM-BASED RECOMMENDATIONS:
1. Frozen Waffles
2. Peanut Butter
3. Cream Cheese
4. Bananas
5. Lettuce


In [58]:
import pandas as pd
import joblib

# -----------------------------
# Load trained item-based model
# -----------------------------
item_similarity = joblib.load("item_similarity.pkl")   # DataFrame
item_list = joblib.load("item_list.pkl")
config = joblib.load("item_cf_config.pkl")

# Override top_n to 3
config["top_n"] = 3

print("Item-based model loaded")

# -----------------------------
# NEW USER DATA
# -----------------------------
new_user_data = pd.DataFrame([
    {"Product_Name": "Organic Bananas", "Rating": 4},
    {"Product_Name": "Vegetables", "Rating": 4},
    {"Product_Name": "Oranges", "Rating": 3}
])

# -----------------------------
# Sentiment function
# -----------------------------
def sentiment_score(rating):
    if rating >= 4:
        return 1
    elif rating == 3:
        return 0
    else:
        return -1

# -----------------------------
# Recommendation logic
# -----------------------------
scores = {}

for _, row in new_user_data.iterrows():
    item = row["Product_Name"]
    if item not in item_similarity.index:
        continue

    weight = (
        config["rating_weight"] * row["Rating"] +
        config["sentiment_weight"] * sentiment_score(row["Rating"])
    )

    similar_items = (
        item_similarity[item]
        .sort_values(ascending=False)
        .iloc[1:config["top_k_items"] + 1]
    )

    for sim_item, sim_score in similar_items.items():
        if sim_item in new_user_data["Product_Name"].values:
            continue
        scores[sim_item] = scores.get(sim_item, 0) + sim_score * weight

# -----------------------------
# TOP-3 RECOMMENDATIONS
# -----------------------------
recommendations = sorted(scores, key=scores.get, reverse=True)[:3]

# -----------------------------
# OUTPUT
# -----------------------------
print("\nNEW USER INPUT:")
print(new_user_data)

print("\nITEM-BASED RECOMMENDATIONS (TOP-3):")
for i, item in enumerate(recommendations, 1):
    print(f"{i}. {item}")

Item-based model loaded

NEW USER INPUT:
      Product_Name  Rating
0  Organic Bananas       4
1       Vegetables       4
2          Oranges       3

ITEM-BASED RECOMMENDATIONS (TOP-3):
1. English Cucumber
2. Honeycrisp Apples
3. Chicken Thighs


User Based Recommendation

IMPORT LIBRARIES

In [95]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import joblib

LOAD DATASET

In [96]:
df = pd.read_csv("grocery_products_dataset.csv")

SENTIMENT STRENGTH

In [97]:
def sentiment_score(r):
    if r >= 4:
        return 1      # positive
    elif r == 3:
        return 0      # neutral
    else:
        return -1     # negative

df['sentiment'] = df['Rating'].apply(sentiment_score)

# sentiment-weighted interaction
df['weighted_rating'] = 0.7 * df['Rating'] + 0.3 * df['sentiment']

FILTER SPARSE USERS

In [98]:
user_counts = df['user_id'].value_counts()
active_users = user_counts[user_counts >= 5].index
df = df[df['user_id'].isin(active_users)]

CREATE USER–ITEM MATRIX

In [99]:
user_item_matrix = df.pivot_table(
    index='user_id',
    columns='Product_Name',
    values='weighted_rating',
    aggfunc='mean'
)

TRAIN USER-BASED MODEL (COSINE SIMILARITY)

In [100]:
user_similarity = pd.DataFrame(
    cosine_similarity(user_item_matrix.fillna(0)),
    index=user_item_matrix.index,
    columns=user_item_matrix.index
)

SAVE THE TRAINED MODEL

In [101]:
joblib.dump(user_similarity, "user_similarity.pkl")
joblib.dump(user_item_matrix.index.tolist(), "user_list.pkl")
joblib.dump(user_item_matrix.columns.tolist(), "item_list.pkl")

config = {
    "rating_weight": 0.7,
    "sentiment_weight": 0.3,
    "similar_users": 5
}
joblib.dump(config, "user_cf_config.pkl")

print("User-based collaborative filtering model trained and saved")

User-based collaborative filtering model trained and saved


In [102]:
joblib.dump(user_item_matrix, "user_item_matrix.pkl")

['user_item_matrix.pkl']

Model Prediction

In [103]:
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics.pairwise import cosine_similarity

# -----------------------------
# Load trained model components
# -----------------------------
user_similarity = joblib.load("user_similarity.pkl")
user_list = joblib.load("user_list.pkl")
item_list = joblib.load("item_list.pkl")
config = joblib.load("user_cf_config.pkl")

# Assuming user_item_matrix was saved in the preceding cell
user_item_matrix = joblib.load("user_item_matrix.pkl")

print("User-based model loaded")

# -----------------------------
# Sentiment function
# -----------------------------
def sentiment_score(rating):
    if rating >= 4:
        return 1
    elif rating == 3:
        return 0
    else:
        return -1

# -----------------------------
# NEW USER DATA (external input)
# -----------------------------
new_user_data = pd.DataFrame([
    {"Product_Name": "Bananas", "Rating": 5},
    {"Product_Name": "Milk", "Rating": 4},
    {"Product_Name": "Frozen Pepperoni Pizza", "Rating": 2}
])

# -----------------------------
# Build new user vector
# -----------------------------
user_vector = pd.Series(0.0, index=item_list)

for _, row in new_user_data.iterrows():
    if row["Product_Name"] in user_vector.index:
        user_vector[row["Product_Name"]] = (
            config["rating_weight"] * row["Rating"] +
            config["sentiment_weight"] * sentiment_score(row["Rating"])
        )

# -----------------------------
# Find similar users
# -----------------------------
similarities = {}

for user in user_list:
    # Compare new user's item vector with existing user's item vector
    existing_user_item_vector = user_item_matrix.loc[user].fillna(0).values
    sim = cosine_similarity(
        [user_vector.values],
        [existing_user_item_vector]
    )[0][0]
    similarities[user] = sim

top_users = sorted(similarities.items(), key=lambda x: x[1], reverse=True)[:5]

# -----------------------------
# Generate recommendations
# -----------------------------
scores = {}

for user, sim in top_users:
    # Get item preferences from user_item_matrix for the similar user
    user_item_preferences = user_item_matrix.loc[user]
    for item, value in user_item_preferences.items():
        if pd.isna(value) or item in new_user_data["Product_Name"].values:
            continue
        scores[item] = scores.get(item, 0) + sim * value

recommendations = sorted(scores, key=scores.get, reverse=True)[:5]

# -----------------------------
# OUTPUT
# -----------------------------
print("\n NEW USER INPUT:")
print(new_user_data)

print("\n RECOMMENDED PRODUCTS:")
for i, item in enumerate(recommendations, 1):
    print(f"{i}. {item}")

User-based model loaded

 NEW USER INPUT:
             Product_Name  Rating
0                 Bananas       5
1                    Milk       4
2  Frozen Pepperoni Pizza       2

 RECOMMENDED PRODUCTS:
1. Turkey Bacon
2. Butter
3. Oat Milk
4. Lettuce
5. Beans


In [104]:
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics.pairwise import cosine_similarity

# -----------------------------
# Load trained model components
# -----------------------------
user_similarity = joblib.load("user_similarity.pkl")
user_list = joblib.load("user_list.pkl")
item_list = joblib.load("item_list.pkl")
config = joblib.load("user_cf_config.pkl")

# Assuming user_item_matrix was saved in the preceding cell
user_item_matrix = joblib.load("user_item_matrix.pkl")

print("User-based model loaded")

# -----------------------------
# Sentiment function
# -----------------------------
def sentiment_score(rating):
    if rating >= 4:
        return 1
    elif rating == 3:
        return 0
    else:
        return -1

# -----------------------------
# NEW USER DATA (external input)
# -----------------------------
new_user_data = pd.DataFrame([
    {"Product_Name": "Milk", "Rating": 4}
])

# -----------------------------
# Build new user vector
# -----------------------------
user_vector = pd.Series(0.0, index=item_list)

for _, row in new_user_data.iterrows():
    if row["Product_Name"] in user_vector.index:
        user_vector[row["Product_Name"]] = (
            config["rating_weight"] * row["Rating"] +
            config["sentiment_weight"] * sentiment_score(row["Rating"])
        )

# -----------------------------
# Find similar users
# -----------------------------
similarities = {}

for user in user_list:
    # Compare new user's item vector with existing user's item vector
    existing_user_item_vector = user_item_matrix.loc[user].fillna(0).values
    sim = cosine_similarity(
        [user_vector.values],
        [existing_user_item_vector]
    )[0][0]
    similarities[user] = sim

top_users = sorted(similarities.items(), key=lambda x: x[1], reverse=True)[:5]

# -----------------------------
# Generate recommendations
# -----------------------------
recommendations_per_product = {}

for _, row in new_user_data.iterrows():
    base_product = row["Product_Name"]
    base_rating = row["Rating"]

    # Skip if product not known
    if base_product not in user_vector.index:
        continue

    scores = {}

    # Use same similar users
    for user, sim in top_users:
        user_item_preferences = user_item_matrix.loc[user]

        for item, value in user_item_preferences.items():
            if pd.isna(value):
                continue
            if item == base_product:
                continue

            # Focus on products liked by similar users
            scores[item] = scores.get(item, 0) + sim * value

    # Take top 5–6 items for THIS product
    recommendations_per_product[base_product] = (
        sorted(scores, key=scores.get, reverse=True)[:5]
    )
print("\n NEW USER INPUT:")
print(new_user_data)

print("\n PRODUCT-WISE RECOMMENDATIONS:")

for product, recs in recommendations_per_product.items():
    print(f"\nBecause you liked '{product}', you may also like:")
    for i, item in enumerate(recs, 1):
        print(f"  {i}. {item}")

User-based model loaded

 NEW USER INPUT:
  Product_Name  Rating
0         Milk       4

 PRODUCT-WISE RECOMMENDATIONS:

Because you liked 'Milk', you may also like:
  1. Soy Milk
  2. Croissants
  3. Zucchini
  4. Chips
  5. Sugar


In [119]:
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics.pairwise import cosine_similarity

# -----------------------------
# Load trained model components
# -----------------------------
user_similarity = joblib.load("user_similarity.pkl")
user_list = joblib.load("user_list.pkl")
item_list = joblib.load("item_list.pkl")
config = joblib.load("user_cf_config.pkl")

# Assuming user_item_matrix was saved in the preceding cell
user_item_matrix = joblib.load("user_item_matrix.pkl")

print("User-based model loaded")

# -----------------------------
# Sentiment function
# -----------------------------
def sentiment_score(rating):
    if rating >= 4:
        return 1
    elif rating == 3:
        return 0
    else:
        return -1

# -----------------------------
# NEW USER DATA (external input)
# -----------------------------
new_user_data = pd.DataFrame([
    {"Product_Name": "Lemons", "Rating": 5}
])

# -----------------------------
# Build new user vector
# -----------------------------
user_vector = pd.Series(0.0, index=item_list)

for _, row in new_user_data.iterrows():
    if row["Product_Name"] in user_vector.index:
        user_vector[row["Product_Name"]] = (
            config["rating_weight"] * row["Rating"] +
            config["sentiment_weight"] * sentiment_score(row["Rating"])
        )

# -----------------------------
# Find similar users
# -----------------------------
similarities = {}

for user in user_list:
    # Compare new user's item vector with existing user's item vector
    existing_user_item_vector = user_item_matrix.loc[user].fillna(0).values
    sim = cosine_similarity(
        [user_vector.values],
        [existing_user_item_vector]
    )[0][0]
    similarities[user] = sim

top_users = sorted(similarities.items(), key=lambda x: x[1], reverse=True)[:5]

# -----------------------------
# Generate recommendations
# -----------------------------
recommendations_per_product = {}

for _, row in new_user_data.iterrows():
    base_product = row["Product_Name"]
    base_rating = row["Rating"]

    # Skip if product not known
    if base_product not in user_vector.index:
        continue

    scores = {}

    # Use same similar users
    for user, sim in top_users:
        user_item_preferences = user_item_matrix.loc[user]

        for item, value in user_item_preferences.items():
            if pd.isna(value):
                continue
            if item == base_product:
                continue

            # Focus on products liked by similar users
            scores[item] = scores.get(item, 0) + sim * value

    # Take top 5–6 items for THIS product
    recommendations_per_product[base_product] = (
        sorted(scores, key=scores.get, reverse=True)[:5]
    )
print("\n NEW USER INPUT:")
print(new_user_data)

print("\n PRODUCT-WISE RECOMMENDATIONS:")

for product, recs in recommendations_per_product.items():
    print(f"\nBecause you liked '{product}', you may also like:")
    for i, item in enumerate(recs, 1):
        print(f"  {i}. {item}")

User-based model loaded

 NEW USER INPUT:
  Product_Name  Rating
0       Lemons       5

 PRODUCT-WISE RECOMMENDATIONS:

Because you liked 'Lemons', you may also like:
  1. Deli Meat
  2. Frozen Berries
  3. Sugar
  4. Turkey Bacon
  5. Popcorn
